# Level 6: Grover's Algorithm
## Q# Edition — Quantum Search

In this notebook, you will:

1. Build the **Oracle** and **Diffusion** operator in Q#
2. Implement Grover's algorithm for a **specific target**
3. Make it **generic** — search for any target
4. Verify with measurement statistics

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. The Oracle: Mark |101⟩

The oracle flips the phase of the target state. For |101⟩, we flip qubits where the target has 0, apply a Controlled Z, then unflip.

In [ ]:
%%qsharp

open Microsoft.Quantum.Diagnostics;

/// Oracle that marks |101⟩ with a phase flip
operation MarkState101(qs : Qubit[]) : Unit is Adj {
    // For target |101⟩: qubit 0 = 1, qubit 1 = 0, qubit 2 = 1
    // Flip qubit 1 (the zero bit) so all qubits are |1⟩
    X(qs[1]);
    // Apply controlled-Z: phase flip when all qubits are |1⟩
    Controlled Z([qs[0], qs[1]], qs[2]);
    // Undo the flip
    X(qs[1]);
}

---
## 2. The Diffusion Operator

Reflects the state about the uniform superposition (inversion about the mean).

In [ ]:
%%qsharp

/// Grover diffusion operator
operation Diffusion(qs : Qubit[]) : Unit is Adj {
    let n = Length(qs);

    // Apply H to all qubits
    ApplyToEach(H, qs);

    // Apply X to all qubits
    ApplyToEach(X, qs);

    // Controlled Z on all qubits
    Controlled Z(qs[0..n-2], qs[n-1]);

    // Undo X and H
    ApplyToEach(X, qs);
    ApplyToEach(H, qs);
}

---
## 3. Full Grover Search

In [ ]:
%%qsharp

/// Run Grover's algorithm to find |101⟩
operation GroverSearch() : Result[] {
    use qs = Qubit[3];

    // Initialize uniform superposition
    ApplyToEach(H, qs);

    // One Grover iteration (optimal for N=8, 1 target)
    MarkState101(qs);
    Diffusion(qs);

    // Second iteration for higher probability
    MarkState101(qs);
    Diffusion(qs);

    // Measure
    let results = MeasureEachZ(qs);
    ResetAll(qs);
    return results;
}

In [ ]:
%%qsharp

/// Run multiple times and count results
operation GroverStatistics() : Unit {
    let shots = 1000;
    mutable found101 = 0;

    for _ in 0..shots - 1 {
        let results = GroverSearch();
        // Check if we got |101⟩
        if results[0] == One and results[1] == Zero and results[2] == One {
            set found101 += 1;
        }
    }

    Message($"Grover search for |101⟩: found {found101}/{shots} times ({found101 * 100 / shots}%)");
    Message($"Random guess would give: {shots / 8}/{shots} (12.5%)");
    Message($"Grover's advantage: {found101 * 100 / shots}% vs 12.5%");
}

GroverStatistics()

In [ ]:
# Visualize the circuit
circuit = qsharp.circuit("GroverSearch()")
Circuit(circuit)

---
## 4. Generic Grover's Algorithm

Let's make it work for any 3-qubit target.

In [ ]:
%%qsharp

/// Generic oracle: marks the state given by targetBits
/// targetBits[i] = true means qubit i should be |1⟩
operation MarkTarget(targetBits : Bool[], qs : Qubit[]) : Unit is Adj {
    let n = Length(qs);

    // Flip qubits where target has 0
    for i in 0..n - 1 {
        if not targetBits[i] {
            X(qs[i]);
        }
    }

    // Multi-controlled Z
    Controlled Z(qs[0..n-2], qs[n-1]);

    // Undo flips
    for i in 0..n - 1 {
        if not targetBits[i] {
            X(qs[i]);
        }
    }
}

/// Generic Grover search for any 3-qubit target
operation GroverGeneric(targetBits : Bool[], iterations : Int) : Result[] {
    let n = Length(targetBits);
    use qs = Qubit[n];

    // Uniform superposition
    ApplyToEach(H, qs);

    // Grover iterations
    for _ in 0..iterations - 1 {
        MarkTarget(targetBits, qs);
        Diffusion(qs);
    }

    let results = MeasureEachZ(qs);
    ResetAll(qs);
    return results;
}

In [ ]:
%%qsharp

operation TestGenericGrover() : Unit {
    // Search for |110⟩
    let target = [true, true, false];
    let shots = 1000;
    mutable found = 0;

    for _ in 0..shots - 1 {
        let results = GroverGeneric(target, 2);
        if results[0] == One and results[1] == One and results[2] == Zero {
            set found += 1;
        }
    }

    Message($"Search for |110⟩: found {found}/{shots} times ({found * 100 / shots}%)");

    // Search for |011⟩
    let target2 = [false, true, true];
    mutable found2 = 0;

    for _ in 0..shots - 1 {
        let results = GroverGeneric(target2, 2);
        if results[0] == Zero and results[1] == One and results[2] == One {
            set found2 += 1;
        }
    }

    Message($"Search for |011⟩: found {found2}/{shots} times ({found2 * 100 / shots}%)");
}

TestGenericGrover()

---
## 5. Key Takeaways

| Concept | Q# Feature |
|---------|------------|
| **Oracle** | `Controlled Z` for multi-controlled phase flip |
| **Diffusion** | H → X → CZ → X → H |
| **ApplyToEach** | Apply a gate to all qubits at once |
| **`is Adj`** | Marks an operation as supporting the Adjoint functor |
| **Array slicing** | `qs[0..n-2]` for control qubits |

---
## 6. Challenges

1. **Optimal iterations**: For 3 qubits (N=8), try 1, 2, and 3 iterations. Which gives the best probability?
2. **4-qubit search**: Extend `GroverGeneric` to work with 4 qubits. How many iterations are needed?
3. **Multiple targets**: Modify the oracle to mark both |101⟩ and |110⟩. Does it still work?

In [ ]:
%%qsharp

// Your challenge code here!

---
**Next up: [Level 7 — Quantum Fourier Transform](../Level_07_QFT/Level_07_QSharp.ipynb)**

# Level 6: Grover's Algorithm
## Q# Edition

In this notebook, you will:

1. Build Grover's search from scratch in Q#
2. Implement the **oracle** and **diffusion** as separate operations
3. Search for a marked state among $2^n$ possibilities

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. Grover's Building Blocks

Two core components:
1. **Oracle**: Flips the phase of the marked state
2. **Diffusion**: Reflects all amplitudes about the mean

In [ ]:
%%qsharp

// Oracle: mark state |101> (we'll flip its phase)
operation MarkState101(qs : Qubit[]) : Unit {
    // Flip qubits that should be |0> in target
    X(qs[1]);  // qubit 1 should be 0 in |101>

    // Apply controlled-Z (phase flip on |111>)
    Controlled Z([qs[0], qs[1]], qs[2]);

    // Undo the X
    X(qs[1]);
}

// Diffusion operator: reflection about |+...+>
operation Diffusion(qs : Qubit[]) : Unit {
    for q in qs { H(q); }
    for q in qs { X(q); }

    // Controlled-Z on all qubits
    Controlled Z(qs[0..Length(qs) - 2], qs[Length(qs) - 1]);

    for q in qs { X(q); }
    for q in qs { H(q); }
}

In [ ]:
%%qsharp

operation GroverSearch(iterations : Int) : Result[] {
    use qs = Qubit[3];

    // Step 1: Uniform superposition
    for q in qs { H(q); }

    // Step 2: Apply Grover iterations
    for _ in 0..iterations - 1 {
        MarkState101(qs);
        Diffusion(qs);
    }

    // Step 3: Measure
    let results = MeasureEachZ(qs);
    ResetAll(qs);
    return results;
}

In [ ]:
# Run with optimal iterations (for 3 qubits: ~2)
print("Single run:", qsharp.eval("GroverSearch(2)"))
print("\nRunning 20 times:")
for i in range(20):
    result = qsharp.eval("GroverSearch(2)")
    bits = ''.join(['1' if r == 'One' else '0' for r in result])
    marker = " ← TARGET" if bits == "101" else ""
    print(f"  Run {i+1:2d}: |{bits}>{marker}")

In [ ]:
# Visualize the circuit
Circuit(qsharp.circuit("GroverSearch(2)"))

### Statistical Verification

In [ ]:
%%qsharp

operation GroverStatistics(iterations : Int, shots : Int) : Int {
    mutable hits = 0;

    for _ in 0..shots - 1 {
        let results = GroverSearch(iterations);
        // Check if we found |101>
        if results[0] == One and results[1] == Zero and results[2] == One {
            set hits += 1;
        }
    }

    return hits;
}

In [ ]:
import matplotlib.pyplot as plt

# Probability vs number of iterations
shots = 5000
probs = []
for iters in range(1, 8):
    hits = qsharp.eval(f"GroverStatistics({iters}, {shots})")
    prob = hits / shots
    probs.append(prob)
    print(f"  {iters} iterations: {prob:.3f} ({hits}/{shots})")

plt.figure(figsize=(10, 5))
plt.bar(range(1, 8), probs, color='darkorange')
plt.axhline(y=1/8, color='r', linestyle='--', label='Random (1/8)')
plt.xlabel('Grover Iterations')
plt.ylabel('Probability of finding |101>')
plt.title("Grover's Algorithm: Q# — Probability vs Iterations")
plt.legend()
plt.show()

---
## 2. Generalized Oracle

Let's make a more flexible version that can search for any 3-qubit target:

In [ ]:
%%qsharp

// Generic oracle: marks any target specified by a Bool array
operation MarkTarget(qs : Qubit[], target : Bool[]) : Unit {
    // Flip qubits where target bit is false (i.e., should be |0>)
    for i in 0..Length(qs) - 1 {
        if not target[i] { X(qs[i]); }
    }

    // Multi-controlled Z
    Controlled Z(qs[0..Length(qs) - 2], qs[Length(qs) - 1]);

    // Undo flips
    for i in 0..Length(qs) - 1 {
        if not target[i] { X(qs[i]); }
    }
}

operation GroverGeneric(target : Bool[], iterations : Int) : Result[] {
    let n = Length(target);
    use qs = Qubit[n];

    for q in qs { H(q); }

    for _ in 0..iterations - 1 {
        MarkTarget(qs, target);
        Diffusion(qs);
    }

    let results = MeasureEachZ(qs);
    ResetAll(qs);
    return results;
}

In [ ]:
# Search for |011>
print("Searching for |011>:")
for i in range(10):
    r = qsharp.eval("GroverGeneric([false, true, true], 2)")
    bits = ''.join(['1' if x == 'One' else '0' for x in r])
    print(f"  Run {i+1}: |{bits}>")

---
## 3. Key Takeaways

| Concept | Description |
|---------|-------------|
| **Grover's** | Quadratic speedup: $O(\sqrt{N})$ |
| **Oracle** | Phase-flips the target state |
| **Diffusion** | Amplifies the marked state |
| **Q# Controlled** | `Controlled Z([controls], target)` for multi-controlled gates |

---
## 4. Challenges

1. **4-qubit Grover**: Extend to 4 qubits, search for |1010>. What's the optimal iteration count?
2. **Multiple solutions**: Modify the oracle to mark two states. How does it affect the iteration count?
3. **Success rate**: Plot success probability for 2, 3, 4, and 5 qubits at optimal iterations.

In [ ]:
%%qsharp

// Your challenge code here!

---
**Next up: [Level 7 — Quantum Fourier Transform](../Level_07_QFT/Level_07_QSharp.ipynb)**